In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_seller_performance
# PURPOSE    : Revenue and performance ranking by seller
# SOURCE     : silver.order_items, silver.orders, silver.sellers
# DESTINATION: fincompliance_catalog.gold.seller_performance
# METRICS:
#   - Total revenue per seller
#   - Total orders per seller
#   - Average price per seller
#   - Seller location (city, state, region)
#   - Seller ranking by revenue
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round as spark_round,
    rank,
    current_timestamp
)
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CELL 5 - READ FROM SILVER
# ============================================================

df_order_items = spark.table(f"{SILVER_DB}.order_items")
df_orders = spark.table(f"{SILVER_DB}.orders")
df_sellers = spark.table(f"{SILVER_DB}.sellers")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"order_items : {df_order_items.count():,} rows")
print(f"orders      : {df_orders.count():,} rows")
print(f"sellers     : {df_sellers.count():,} rows")

print("\nsellers columns:")
for c in df_sellers.columns:
    print(f"  {c}")

In [0]:
# ============================================================
# CALCULATE SELLER PERFORMANCE
# Join order_items with delivered orders and sellers
# Group by seller with location details
# ============================================================

df_seller_performance = (
    df_order_items
    .join(
        df_orders.filter(col("order_status") == "delivered")
                  .select("order_id"),
        on="order_id",
        how="inner"
    )
    .join(
        df_sellers.select(
            "seller_id",
            "seller_city",
            "seller_state",
            "seller_state_name",
            "seller_region"
        ),
        on="seller_id",
        how="left"
    )
    .groupBy(
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_state_name",
        "seller_region"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("order_item_id").alias("total_units_sold"),
        spark_round(spark_sum("price"), 2)
            .alias("total_revenue"),
        spark_round(avg("price"), 2)
            .alias("avg_price"),
        spark_round(avg("freight_value"), 2)
            .alias("avg_freight")
    )
    .orderBy(col("total_revenue").desc())
)

print("Top 10 sellers by revenue:")
df_seller_performance.show(10, truncate=False)

print(f"\nTotal sellers : {df_seller_performance.count():,}")

In [0]:
# ============================================================
# ADD SELLER RANKING
# ============================================================

window_rank = Window.orderBy(col("total_revenue").desc())

df_seller_performance = df_seller_performance \
    .withColumn(
        "revenue_rank",
        rank().over(window_rank)
    )

print("Top 10 sellers with rank:")
df_seller_performance \
    .select(
        "revenue_rank",
        "seller_id",
        "seller_state",
        "total_revenue",
        "total_orders"
    ) \
    .orderBy("revenue_rank") \
    .show(10, truncate=False)

# Show revenue by region
print("\nTotal revenue by seller region:")
df_seller_performance \
    .groupBy("seller_region") \
    .agg(
        spark_sum("total_revenue").alias("region_revenue"),
        countDistinct("seller_id").alias("seller_count")
    ) \
    .orderBy(col("region_revenue").desc()) \
    .show(truncate=False)

print("Done - Ranking and region summary added")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_seller_performance = df_seller_performance \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_seller_performance.columns:
    print(f"  {col_name}")
print(f"Total sellers : {df_seller_performance.count():,}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_seller_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.seller_performance")
)

print(f"Written to {GOLD_DB}.seller_performance")
print(f"Rows : {df_seller_performance.count():,}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.SELLER_PERFORMANCE VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.seller_performance")
print(f"Total sellers : {df_gold.count():,}")
print(f"Total columns : {len(df_gold.columns)}")

print("\nRevenue by region (properly formatted):")
df_gold \
    .groupBy("seller_region") \
    .agg(
        spark_round(spark_sum("total_revenue"), 2)
            .alias("region_revenue"),
        countDistinct("seller_id").alias("seller_count")
    ) \
    .orderBy(col("region_revenue").desc()) \
    .show(truncate=False)

print("\nTop 3 sellers:")
df_gold.orderBy("revenue_rank").show(3, truncate=False)

print("=" * 60)
print("gold.seller_performance verification complete!")
print("=" * 60)